# Stage 1a: Kinetic analysis - exponential and initial-rate workflow (exponential increase)

This notebook analyzes kinetic absorbance data from plate readers. It imports the data, generates an overview plot, performs monoexponential fitting, calculates initial-rate-derived rate constants for slow or unreliable exponential traces, compares both approaches, and exports diagnostic plots and fitted parameters.

To use the notebook, place it in the main kinetics directory alongside `P5`, `P6`, ..., `P9`, then provide the phosphorylating-agent directory, experiment subfolder, and raw Excel filename in the configuration cell below. All output files are written automatically to an analysis output folder inside the experiment directory.

This file can handle the raw data (exported as .xlsx) from ThermoFisher Multiskan v6, Agilent/Biotek Gen 5.5, or BMG Labtech readers.

## Part I: Configuration

Define the phosphorylating-agent directory, project folder, raw Excel file, output location, analysis scope, and fitting parameters. In most cases, only the first three settings need to be changed.

In [ ]:
from pathlib import Path

# =========================
# USER INPUT
# =========================
# Put this notebook in the main kinetics directory, e.g.:
# kinetics/
#   P7/
#     S4_NS062/
#       NS062-mes_NS063.xlsx
#
# Then only edit these three lines:

PHOSPHORYLATING_AGENT_DIR = "P7"
PROJECT_FOLDER_NAME = "S4_NS062"
RAW_EXCEL_NAME = "NS062-mes_NS063.xlsx"

# PATH SETUP
# Main directory containing the P5-P9 folders.
KINETICS_DIR = Path.cwd()

# Folder for this specific phosphorylating agent and experiment/dataset.
BASE_DIR = KINETICS_DIR / PHOSPHORYLATING_AGENT_DIR / PROJECT_FOLDER_NAME

# Raw plate-reader export.
input_excel = BASE_DIR / RAW_EXCEL_NAME

# Output location.
output_dir = BASE_DIR / "exp_initial_rate_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# Output files.
overview_png = output_dir / "kinetics_overview.png"
fit_results_excel = output_dir / f"{PROJECT_FOLDER_NAME}_exp_initial_rate_fit_results.xlsx"

# Set to None to analyze the full plate. For quick tests, use e.g. ["D4", "D5", "D6", "D7", "D8", "D9", "D10", "D11"].
wells_to_analyze = None

# Exponential-fit settings copied from the current analysis routine.
maxfev_value = 200000 #200000
r2_tolerance = 0.0050
exp_max_start_time_adjustment_factor = 0.03 #0.03
exp_conversion = 0.80 #0.80

# Initial-rate settings.
linear_r2_tolerance = 0.0050 # This is the tolerance factor based on the R^2 value based on which the longes fit is selected.
minimum_linear_region_fraction = 0.50 # This is the minimum fraction of the total time course that must be included in the linear region.
linear_start_fraction_candidates = [0.00, 0.005, 0.01, 0.02, 0.03, 0.05] # For the initial rate analysis, these are possible start fractions.
linear_end_fraction_candidates = [1.00, 0.75, 0.50] # For the initial rate analysis, these are possible total conversions.
initial_rate_exp_a_relative_error_cutoff = 0.10 # This is the relative error cutoff for the exponential fit parameter 'a'.

# Exponential reliability filters. These do not change the exponential fitting routine itself;
# they decide whether a returned exponential fit is trustworthy enough for downstream use.
exp_k_relative_error_cutoff = 0.50
exp_max_plausible_k = 1e-4  # s^-1; faster fits are treated as early artifacts/local transients. Set to None to disable. Default: 1e-4.

# Plot settings.
save_plots = True
show_plots = True

# Plate-level filtering for the initial rate analysis.
a_relative_error_cutoff = 0.10
minimum_reliable_a_values = 10
bootstrap_iterations = 10000
bootstrap_random_seed = 42

print(f"Input Excel: {input_excel}")
print(f"Output directory: {output_dir}")


## Part II: Import raw plate data

Import the raw plate-reader Excel file and automatically detect the format (e.g., Thermo or Biotek). The data are converted into one standardized table with `Time` and normalized well names.

In [ ]:
import math
import re
import numpy as np
import pandas as pd


def convert_time_to_seconds(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, str):
        try:
            return pd.to_timedelta(val).total_seconds()
        except Exception:
            try:
                return float(val)
            except Exception:
                return np.nan
    if isinstance(val, pd.Timestamp):
        return val.hour * 3600 + val.minute * 60 + val.second + val.microsecond / 1e6
    if isinstance(val, pd.Timedelta):
        return val.total_seconds()
    if isinstance(val, (int, float, np.integer, np.floating)):
        return float(val)
    if hasattr(val, "hour") and hasattr(val, "minute") and hasattr(val, "second"):
        return val.hour * 3600 + val.minute * 60 + val.second
    return np.nan


def normalize_well_name(name, plate_columns=24):
    if pd.isna(name):
        return name
    text = str(name).strip()

    paren_match = re.search(r"\(([A-Za-z]+)0*(\d+)\)", text)
    if paren_match:
        return f"{paren_match.group(1).upper()}{int(paren_match.group(2))}"

    direct_match = re.fullmatch(r"([A-Za-z]+)0*(\d+)", text)
    if direct_match:
        return f"{direct_match.group(1).upper()}{int(direct_match.group(2))}"

    un_match = re.fullmatch(r"Un0*(\d+)", text, flags=re.IGNORECASE)
    if un_match:
        well_number = int(un_match.group(1))
        row_index = (well_number - 1) // plate_columns
        col_index = (well_number - 1) % plate_columns + 1
        row_label = chr(ord("A") + row_index)
        return f"{row_label}{col_index}"

    return text


def clean_export_table(df):
    """Convert blank strings from plate-reader exports to NaN before parsing."""
    with pd.option_context("future.no_silent_downcasting", True):
        cleaned = df.replace(r"^\s*$", np.nan, regex=True)
    return cleaned.infer_objects(copy=False)


def standardize_plate_dataframe(df):
    df = clean_export_table(df.copy())
    rename_map = {}
    for column in df.columns:
        if isinstance(column, str) and column.strip().lower() in ["time", "avg. time [s]"]:
            rename_map[column] = "Time"
        elif column != "Time":
            rename_map[column] = normalize_well_name(column)
    df = df.rename(columns=rename_map)
    df = df.loc[:, df.columns.notna()]
    df = df.loc[:, ~df.columns.duplicated()]

    if "Reading" in df.columns:
        df = df.drop(columns=["Reading"])
    if "Time" not in df.columns:
        raise ValueError("Imported table does not contain a Time column.")

    for column in df.columns:
        if column == "Time":
            df[column] = df[column].apply(convert_time_to_seconds)
        else:
            df[column] = pd.to_numeric(df[column], errors="coerce")

    df = df.dropna(axis=0, how="all")
    df = df.dropna(axis=1, how="all")
    df = df.dropna(subset=["Time"])
    return df


def concatenate_sequential_blocks(blocks):
    adjusted_blocks = []
    time_offset = 0.0

    for block in blocks:
        block = block.copy()
        block = standardize_plate_dataframe(block)
        block = block.sort_values("Time").reset_index(drop=True)
        block["Time"] = block["Time"] + time_offset
        adjusted_blocks.append(block)

        finite_time = block["Time"].dropna()
        if not finite_time.empty:
            time_offset = float(finite_time.iloc[-1])

    if not adjusted_blocks:
        raise ValueError("No readable kinetic data blocks were found.")

    combined = pd.concat(adjusted_blocks, ignore_index=True, sort=False)
    combined = combined.loc[:, ~combined.columns.duplicated()]
    well_columns = [column for column in combined.columns if column != "Time"]
    combined = combined.dropna(subset=["Time"])
    combined = combined.dropna(subset=well_columns, how="all").reset_index(drop=True)
    return combined


def assemble_synergy_segments(blocks):
    segments = []
    current_segment = None
    time_offset = 0.0

    for block in blocks:
        block = standardize_plate_dataframe(block)
        block = block.sort_values("Time").reset_index(drop=True)

        well_columns = [column for column in block.columns if column != "Time"]
        if not well_columns:
            continue
        first_well = well_columns[0]

        starts_new_segment = current_segment is None or first_well == "A1"
        if starts_new_segment:
            if current_segment is not None:
                segments.append(current_segment)
                finite_time = current_segment["Time"].dropna()
                if not finite_time.empty:
                    time_offset = float(finite_time.iloc[-1])

            current_segment = block.copy()
            current_segment["Time"] = current_segment["Time"] + time_offset
        else:
            block = block.copy()
            block["Time"] = block["Time"] + time_offset
            current_segment = pd.merge(current_segment, block, on="Time", how="outer")
            current_segment = current_segment.loc[:, ~current_segment.columns.duplicated()]

    if current_segment is not None:
        segments.append(current_segment)

    if not segments:
        raise ValueError("No readable Synergy kinetic data segments were found.")

    combined = pd.concat(segments, ignore_index=True, sort=False)
    combined = combined.sort_values("Time").reset_index(drop=True)
    well_columns = [column for column in combined.columns if column != "Time"]
    combined = combined.dropna(subset=["Time"])
    combined = combined.dropna(subset=well_columns, how="all").reset_index(drop=True)
    return combined


def find_skanit_reading_rows(df_full):
    text = df_full.astype(str).apply(lambda column: column.str.strip())
    reading_mask = text.eq("Reading")
    reading_rows = sorted(set(int(row) for row in np.where(reading_mask)[0]))
    return reading_rows


def sheet_has_skanit_table(df_full):
    reading_rows = find_skanit_reading_rows(df_full)
    if not reading_rows:
        return False

    for header_row in reading_rows:
        header_values = [
            str(value).strip().lower()
            for value in df_full.iloc[header_row].dropna().tolist()
        ]
        if "reading" in header_values and "avg. time [s]" in header_values:
            return True
    return False


def detect_reader_format(input_excel):
    xls = pd.ExcelFile(input_excel)
    absorbance_sheets = [
        sheet for sheet in xls.sheet_names
        if str(sheet).lower().startswith("absorbance")
    ]

    if absorbance_sheets:
        return "skanit_absorbance_sheets", xls

    for sheet_name in xls.sheet_names:
        df_full = pd.read_excel(xls, sheet_name=sheet_name, header=None, nrows=200)
        if sheet_has_skanit_table(df_full):
            return "skanit_absorbance_sheets", xls

    for sheet_name in xls.sheet_names:
        df_full = pd.read_excel(xls, sheet_name=sheet_name, header=None, nrows=200)
        if df_full.shape[1] > 1 and (df_full.iloc[:, 1] == "Time").any():
            return "synergy_time_blocks", xls

    return "unknown", xls


def read_skanit_absorbance_sheets(xls):
    blocks = []
    source_tables = []
    absorbance_sheets = [
        sheet for sheet in xls.sheet_names
        if str(sheet).lower().startswith("absorbance")
    ]

    if absorbance_sheets:
        candidate_sheets = absorbance_sheets
    else:
        candidate_sheets = []
        for sheet_name in xls.sheet_names:
            df_preview = pd.read_excel(xls, sheet_name=sheet_name, header=None, nrows=200)
            if sheet_has_skanit_table(df_preview):
                candidate_sheets.append(sheet_name)

    for sheet_name in candidate_sheets:
        df_full = pd.read_excel(xls, sheet_name=sheet_name, header=None)
        df_full = clean_export_table(df_full)
        reading_rows = find_skanit_reading_rows(df_full)
        if not reading_rows:
            continue

        for table_index, header_row in enumerate(reading_rows):
            next_header_row = reading_rows[table_index + 1] if table_index + 1 < len(reading_rows) else len(df_full)
            block = df_full.iloc[header_row + 1:next_header_row].copy()
            block.columns = df_full.iloc[header_row]
            block = block.loc[:, block.columns.notna()]
            block = clean_export_table(block)
            block = block.dropna(axis=0, how="all")
            block = block.dropna(axis=1, how="all")
            if block.empty:
                continue
            blocks.append(block)
            source_tables.append(f"{sheet_name}:table{table_index + 1}")

    combined = concatenate_sequential_blocks(blocks)
    summary = {
        "detected_format": "skanit_absorbance_sheets",
        "blocks_imported": len(blocks),
        "source_sheets": candidate_sheets,
        "source_tables": source_tables,
    }
    return combined, summary


def trim_synergy_kinetic_rows(block):
    """Keep only the initial kinetic rows before Synergy footer/layout rows begin."""
    if block.empty:
        return block

    time_column = block.columns[0]
    converted_time = block[time_column].apply(convert_time_to_seconds)
    keep_count = 0
    previous_time = -np.inf

    for value in converted_time:
        if not np.isfinite(value):
            break
        if keep_count > 0 and value <= previous_time:
            break
        keep_count += 1
        previous_time = value

    return block.iloc[:keep_count].copy()


def read_synergy_time_blocks(xls):
    blocks = []
    source_blocks = []

    for sheet_name in xls.sheet_names:
        df_full = pd.read_excel(xls, sheet_name=sheet_name, header=None)
        if df_full.shape[1] < 2:
            continue

        time_rows = df_full.index[df_full.iloc[:, 1] == "Time"].tolist()
        for block_number, header_row in enumerate(time_rows, start=1):
            block = df_full.iloc[header_row + 1:].copy()
            block.columns = df_full.iloc[header_row]
            block = block.loc[:, block.columns.notna()]

            next_time_rows = block.index[block.iloc[:, 0] == "Time"].tolist()
            if next_time_rows:
                block = block.loc[:next_time_rows[0] - 1]

            block = block.dropna(axis=0, how="all")
            block = block.dropna(axis=1, how="all")

            # Thermo/Synergy exports often put temperature immediately after time.
            cols = list(block.columns)
            if len(cols) > 1 and str(cols[1]).strip().lower() not in ["a1", "a01"]:
                block = block.drop(columns=[cols[1]])

            block = trim_synergy_kinetic_rows(block)
            if block.empty:
                continue

            blocks.append(block)
            source_blocks.append(f"{sheet_name}:block{block_number}")

    combined = assemble_synergy_segments(blocks)
    summary = {
        "detected_format": "synergy_time_blocks",
        "blocks_imported": len(blocks),
        "source_sheets": source_blocks,
    }
    return combined, summary


def read_plate_reader_excel(input_excel):
    reader_format, xls = detect_reader_format(input_excel)

    if reader_format == "skanit_absorbance_sheets":
        data, summary = read_skanit_absorbance_sheets(xls)
    elif reader_format == "synergy_time_blocks":
        data, summary = read_synergy_time_blocks(xls)
    else:
        raise ValueError(
            "Could not detect a supported plate-reader export format. "
            "Expected Absorbance sheets or Synergy-style Time blocks."
        )

    if "Time" not in data.columns:
        raise ValueError("Import failed: no Time column was produced.")

    well_columns = [column for column in data.columns if column != "Time"]
    if not well_columns:
        raise ValueError("Import failed: no well columns were produced.")

    if not data["Time"].is_monotonic_increasing:
        data = data.sort_values("Time").reset_index(drop=True)

    summary.update(
        {
            "n_rows": len(data),
            "n_wells": len(well_columns),
            "time_min": float(data["Time"].min()),
            "time_max": float(data["Time"].max()),
        }
    )
    return data, summary


data_frame, import_summary = read_plate_reader_excel(input_excel)
data_frame = data_frame.replace("#ERR", np.nan)

for column in data_frame.columns:
    data_frame[column] = pd.to_numeric(data_frame[column], errors="coerce")

well_columns = [column for column in data_frame.columns if column != "Time"]
data_frame = data_frame.dropna(subset=["Time"])
data_frame = data_frame.dropna(subset=well_columns, how="all").reset_index(drop=True)

if wells_to_analyze is None:
    analysis_columns = list(data_frame.columns[1:])
else:
    missing_wells = [well for well in wells_to_analyze if well not in data_frame.columns]
    if missing_wells:
        raise ValueError(f"Requested wells were not found in the data: {missing_wells}")
    analysis_columns = list(wells_to_analyze)

print("Final combined DataFrame:")
print("Import summary:")
for key, value in import_summary.items():
    if key == "source_sheets" and isinstance(value, list):
        print(f"  {key}: {value[:6]}{'...' if len(value) > 6 else ''}")
    else:
        print(f"  {key}: {value}")
print(data_frame.head())
print(data_frame.tail())
print(f"Shape: {data_frame.shape}")
print(f"Analysis columns: {len(analysis_columns)} wells")


## Part III: Overview plot

This step generate a plate-wide overview plot of all kinetic traces. This plot is intended for quick visual inspection before evaluating individual fits.

In [ ]:
import matplotlib.pyplot as plt


columns_starting_with_a = [
    col for col in data_frame.columns
    if isinstance(col, str) and col.startswith("A")
]
num_cols = len(columns_starting_with_a)
if num_cols == 0:
    num_cols = 12

num_plots = len(data_frame.columns) - 1
num_rows = math.ceil(num_plots / num_cols)

fig, axes = plt.subplots(num_rows, num_cols, figsize=(30, 2 * num_rows))
axes = np.atleast_2d(axes)
fig.subplots_adjust(hspace=0.8, wspace=0.8)

for i, column in enumerate(data_frame.columns[1:]):
    row_idx = i // num_cols
    col_idx = i % num_cols

    axes[row_idx, col_idx].plot(data_frame["Time"], data_frame[column], label=column)
    axes[row_idx, col_idx].set_title(column)
    axes[row_idx, col_idx].set_xlabel("Time")
    axes[row_idx, col_idx].set_ylabel("Absorbance")
    axes[row_idx, col_idx].grid(True)
    axes[row_idx, col_idx].tick_params(
        axis="both", which="both",
        bottom=False, top=False, left=False, right=False,
        labelbottom=False, labelleft=False,
    )

for i in range(num_plots, num_cols * num_rows):
    axes.flatten()[i].axis("off")

plt.savefig(overview_png, dpi=300, bbox_inches="tight")
plt.show()

print(f"Overview plot saved to {overview_png}")


## Part IV: Shared fitting helpers

Define the exponential model, linear fallback model, reader-independent helper functions, and reliability checks used throughout the analysis.

In [ ]:
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score


def exponential_increase_with_offset(x, A, k, C):
    return C + A * (1 - np.exp(-k * x))


def linear_model(x, slope, intercept):
    return slope * x + intercept


def calculate_time_regions(data_length, min_points=10):
    time_regions = []
    region = 1.0
    while True:
        final_index = int(region * data_length)
        if final_index < min_points:
            break
        time_regions.append(round(region, 5))
        region *= 0.5
    return sorted(time_regions)


def parameter_errors(covariance, n_params):
    if covariance is None:
        return [np.nan] * n_params
    try:
        diag = np.diag(covariance)
        diag = np.where(diag >= 0, diag, np.nan)
        return np.sqrt(diag).tolist()
    except Exception:
        return [np.nan] * n_params


def relative_error(value, error):
    if np.isfinite(value) and value != 0 and np.isfinite(error):
        return abs(error / value)
    return np.nan


def classify_exponential_fit(fit):
    reasons = []

    if not fit["success"]:
        return False, fit.get("failure_reason") or "fit_failed"

    for key in ["A", "k", "C", "A_err", "k_err", "C_err", "R2", "t_conv"]:
        if not np.isfinite(fit.get(key, np.nan)):
            reasons.append(f"{key}_not_finite")

    if np.isfinite(fit.get("k", np.nan)) and fit["k"] <= 0:
        reasons.append("k_not_positive")

    if (
        exp_max_plausible_k is not None
        and np.isfinite(exp_max_plausible_k)
        and np.isfinite(fit.get("k", np.nan))
        and fit["k"] > exp_max_plausible_k
    ):
        reasons.append("k_above_max_plausible")

    a_rel = relative_error(fit.get("A", np.nan), fit.get("A_err", np.nan))
    k_rel = relative_error(fit.get("k", np.nan), fit.get("k_err", np.nan))

    if not np.isfinite(a_rel) or a_rel >= a_relative_error_cutoff:
        reasons.append("A_relative_error_too_high")
    if not np.isfinite(k_rel) or k_rel >= exp_k_relative_error_cutoff:
        reasons.append("k_relative_error_too_high")

    if len(fit.get("x_precise", [])) < 3:
        reasons.append("too_few_precise_fit_points")

    return len(reasons) == 0, "; ".join(reasons)


def safe_r2(y_true, y_pred):
    try:
        if len(y_true) < 2:
            return np.nan
        return r2_score(y_true, y_pred)
    except Exception:
        return np.nan


def fit_exponential_current_routine(
    x,
    y,
    conversion=0.80,
    max_start_time_adjustment_factor=0.03,
    min_points=10,
):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    result = {
        "success": False,
        "failure_reason": None,
        "A": np.nan,
        "k": np.nan,
        "C": np.nan,
        "A_err": np.nan,
        "k_err": np.nan,
        "C_err": np.nan,
        "R2": np.nan,
        "chosen_region": np.nan,
        "chosen_fit_start_time": np.nan,
        "chosen_fit_end_time": np.nan,
        "precise_fit_start_time": np.nan,
        "precise_fit_end_time": np.nan,
        "t_conv": np.nan,
        "x_precise": np.array([]),
        "y_precise": np.array([]),
        "y_precise_pred": np.array([]),
        "params": None,
    }

    if len(x) < max(3, min_points):
        result["failure_reason"] = "insufficient_data"
        return result

    time_regions = calculate_time_regions(len(x), min_points=min_points)
    best_region_length = 0
    best_params = None
    best_region = None
    best_r2 = -float("inf")

    for region in time_regions:
        try:
            final_index = int(region * len(x))
            x_fit = x[:final_index]
            y_fit = y[:final_index]

            if len(y_fit) < 3:
                continue

            initial_guesses = [y_fit[-1] - y_fit[0], 0.00001, y_fit[0]]
            params, covariance = curve_fit(
                exponential_increase_with_offset,
                x_fit,
                y_fit,
                p0=initial_guesses,
                maxfev=maxfev_value,
            )
            y_pred = exponential_increase_with_offset(x_fit, *params)
            r2 = safe_r2(y_fit, y_pred)

            if final_index > best_region_length and r2 >= best_r2 - r2_tolerance:
                best_region_length = final_index
                best_params = params
                best_r2 = r2
                best_region = region

        except Exception:
            continue

    if best_region is None:
        result["failure_reason"] = "no_valid_region"
        return result

    try:
        custom_region_index = best_region_length
        x_custom_fit = x[:custom_region_index]
        y_custom_fit = y[:custom_region_index]

        params_custom, _ = curve_fit(
            exponential_increase_with_offset,
            x_custom_fit,
            y_custom_fit,
            p0=best_params,
            maxfev=maxfev_value,
        )

        A_custom, k_custom, C_custom = params_custom
        result["t_conv"] = np.log(1 / (1 - conversion)) / k_custom

        if not np.isfinite(result["t_conv"]) or result["t_conv"] < 0:
            result["failure_reason"] = "negative_or_invalid_t_conv"
            return result

        closest_time_point = x[np.abs(x - result["t_conv"]).argmin()]
        precise_fit_end_time = closest_time_point

        max_start_time_adjustment = max_start_time_adjustment_factor * precise_fit_end_time
        possible_indices = np.where(
            x_custom_fit <= min(x_custom_fit[-1], x[0] + max_start_time_adjustment)
        )[0]
        if len(possible_indices) == 0:
            precise_fit_start_index = 0
        else:
            precise_fit_start_index = max(possible_indices[-1], 0)

        precise_fit_start_time = x[precise_fit_start_index]
        precise_fit_indices = np.where(
            (x_custom_fit >= precise_fit_start_time)
            & (x_custom_fit <= precise_fit_end_time)
        )[0]
        precise_x_custom_fit = x_custom_fit[precise_fit_indices]
        precise_y_custom_fit = y_custom_fit[precise_fit_indices]

        if len(precise_x_custom_fit) >= 3 and len(precise_y_custom_fit) >= 3:
            params_precise, covariance_precise = curve_fit(
                exponential_increase_with_offset,
                precise_x_custom_fit,
                precise_y_custom_fit,
                p0=params_custom,
                maxfev=maxfev_value,
            )
            y_precise_fit_pred = exponential_increase_with_offset(
                precise_x_custom_fit,
                *params_precise,
            )
            precise_r2 = safe_r2(precise_y_custom_fit, y_precise_fit_pred)
        else:
            params_precise = params_custom
            covariance_precise = None
            precise_x_custom_fit = x_custom_fit
            precise_y_custom_fit = y_custom_fit
            y_precise_fit_pred = exponential_increase_with_offset(
                precise_x_custom_fit,
                *params_precise,
            )
            precise_r2 = best_r2

        A, k, C = params_precise
        A_err, k_err, C_err = parameter_errors(covariance_precise, 3)

        result.update(
            {
                "success": True,
                "A": A,
                "k": k,
                "C": C,
                "A_err": A_err,
                "k_err": k_err,
                "C_err": C_err,
                "R2": precise_r2,
                "chosen_region": best_region,
                "chosen_fit_start_time": x[0],
                "chosen_fit_end_time": x[custom_region_index - 1],
                "precise_fit_start_time": precise_fit_start_time,
                "precise_fit_end_time": precise_fit_end_time,
                "x_precise": np.atleast_1d(precise_x_custom_fit),
                "y_precise": np.atleast_1d(precise_y_custom_fit),
                "y_precise_pred": np.atleast_1d(y_precise_fit_pred),
                "params": params_precise,
            }
        )
        return result

    except Exception as exc:
        result["failure_reason"] = str(exc)
        return result


def fit_linear_roi(
    x,
    y,
    min_region_fraction=0.50,
    start_fraction_candidates=None,
    end_fraction_candidates=None,
):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    result = {
        "success": False,
        "failure_reason": None,
        "slope": np.nan,
        "intercept": np.nan,
        "slope_err": np.nan,
        "intercept_err": np.nan,
        "R2": np.nan,
        "chosen_region": np.nan,
        "chosen_start_fraction": np.nan,
        "chosen_end_fraction": np.nan,
        "fit_start_time": np.nan,
        "fit_end_time": np.nan,
        "x_fit": np.array([]),
        "y_fit": np.array([]),
        "y_pred": np.array([]),
    }

    if len(x) < 2:
        result["failure_reason"] = "insufficient_data"
        return result

    if start_fraction_candidates is None:
        start_fraction_candidates = [0.0]
    if end_fraction_candidates is None:
        end_fraction_candidates = [1.0, 0.5]

    candidates = []

    for end_fraction in sorted(end_fraction_candidates, reverse=True):
        if end_fraction < min_region_fraction:
            continue
        end_index = int(end_fraction * len(x))
        end_index = min(max(end_index, 2), len(x))

        for start_fraction in sorted(start_fraction_candidates):
            if start_fraction < 0 or start_fraction >= end_fraction:
                continue

            start_index = int(start_fraction * len(x))
            window_length = end_index - start_index
            if window_length < 2:
                continue
            if window_length / len(x) < min_region_fraction:
                continue

            x_fit = x[start_index:end_index]
            y_fit = y[start_index:end_index]

            if len(y_fit) < 2 or x_fit[-1] == x_fit[0]:
                continue

            try:
                initial_guesses = [(y_fit[-1] - y_fit[0]) / (x_fit[-1] - x_fit[0]), y_fit[0]]
                params, covariance = curve_fit(
                    linear_model,
                    x_fit,
                    y_fit,
                    p0=initial_guesses,
                    maxfev=maxfev_value,
                )
                y_pred = linear_model(x_fit, *params)
                r2 = safe_r2(y_fit, y_pred)

                candidates.append(
                    {
                        "window_length": window_length,
                        "params": params,
                        "covariance": covariance,
                        "r2": r2,
                        "start_fraction": start_fraction,
                        "end_fraction": end_fraction,
                        "start_index": start_index,
                        "end_index": end_index,
                    }
                )

            except Exception:
                continue

    if not candidates:
        result["failure_reason"] = "no_valid_region"
        return result

    finite_r2_values = [candidate["r2"] for candidate in candidates if np.isfinite(candidate["r2"])]
    if not finite_r2_values:
        result["failure_reason"] = "no_finite_r2"
        return result

    best_r2 = max(finite_r2_values)
    acceptable_candidates = [
        candidate for candidate in candidates
        if np.isfinite(candidate["r2"]) and candidate["r2"] >= best_r2 - linear_r2_tolerance
    ]
    if not acceptable_candidates:
        result["failure_reason"] = "no_acceptable_region"
        return result

    best_candidate = sorted(
        acceptable_candidates,
        key=lambda candidate: (
            candidate["window_length"],
            candidate["start_fraction"],
            candidate["end_fraction"],
        ),
        reverse=True,
    )[0]

    best_window_length = best_candidate["window_length"]
    best_params = best_candidate["params"]
    best_covariance = best_candidate["covariance"]
    best_r2 = best_candidate["r2"]
    best_start_fraction = best_candidate["start_fraction"]
    best_end_fraction = best_candidate["end_fraction"]
    best_start_index = best_candidate["start_index"]
    best_end_index = best_candidate["end_index"]

    x_fit = x[best_start_index:best_end_index]
    y_fit = y[best_start_index:best_end_index]
    slope, intercept = best_params
    y_pred = linear_model(x_fit, slope, intercept)
    slope_err, intercept_err = parameter_errors(best_covariance, 2)

    result.update(
        {
            "success": True,
            "slope": slope,
            "intercept": intercept,
            "slope_err": slope_err,
            "intercept_err": intercept_err,
            "R2": best_r2,
            "chosen_region": best_window_length / len(x),
            "chosen_start_fraction": best_start_fraction,
            "chosen_end_fraction": best_end_fraction,
            "fit_start_time": x_fit[0],
            "fit_end_time": x_fit[-1],
            "x_fit": x_fit,
            "y_fit": y_fit,
            "y_pred": y_pred,
        }
    )
    return result


## Part V: Exponential analysis

Fit each kinetic trace with a monoexponential fitting routine and save diagnostic plots. Fit parameters, errors, selected fit regions, and reliability flags are stored for later comparison.

In [ ]:
def plot_exponential_fit(column, x, y, fit, output_path, prefix_label="Exponential"):
    fig, (ax1, ax2, ax3) = plt.subplots(
        3, 1, figsize=(8, 15), gridspec_kw={"height_ratios": [3, 3, 1]}
    )
    fig.suptitle(f"{prefix_label} analysis for {column}", y=0.925)

    ax1.plot(x, y, label="Original Data")
    if fit["success"]:
        ax1.plot(
            fit["x_precise"],
            fit["y_precise_pred"],
            label=(
                f"Precise Fit (A={fit['A']:.4f}, k={fit['k']:.7f}, "
                f"C={fit['C']:.4f}, R2={fit['R2']:.4f})"
            ),
            color="red",
        )
        ax1.fill_betweenx(
            ax1.get_ylim(),
            fit["precise_fit_start_time"],
            fit["precise_fit_end_time"],
            color="gray",
            alpha=0.2,
            label="Precise Fit Region",
        )
    ax1.set_xlabel("Time [s]")
    ax1.set_ylabel("Value")
    ax1.legend()

    ax2.plot(x, y, label="Original Data")
    if fit["success"]:
        ax2.plot(
            fit["x_precise"],
            fit["y_precise_pred"],
            label=(
                f"Precise Fit (A={fit['A']:.4f}, k={fit['k']:.7f}, "
                f"C={fit['C']:.4f}, R2={fit['R2']:.4f})"
            ),
            color="red",
        )
        ax2.fill_betweenx(
            ax2.get_ylim(),
            fit["precise_fit_start_time"],
            fit["precise_fit_end_time"],
            color="gray",
            alpha=0.2,
            label="Precise Fit Region",
        )
        ax2.set_xlim(fit["precise_fit_start_time"], fit["precise_fit_end_time"])
    ax2.set_xlabel("Time [s]")
    ax2.set_ylabel("Value")
    ax2.legend()

    if fit["success"]:
        residuals = fit["y_precise"] - fit["y_precise_pred"]
        ax3.plot(fit["x_precise"], residuals, label="Residuals", color="purple")
        ax3.set_xlim(fit["precise_fit_start_time"], fit["precise_fit_end_time"])
    ax3.axhline(y=0, color="black", linestyle="--")
    ax3.set_xlabel("Time [s]")
    ax3.set_ylabel("Residual")
    ax3.legend()

    if save_plots:
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
    if show_plots:
        plt.show()
    else:
        plt.close(fig)


exp_fit_results = {}
exp_records = []

x_all = data_frame["Time"].values

for column in analysis_columns:
    x = data_frame["Time"].values
    y = data_frame[column].values

    fit = fit_exponential_current_routine(
        x,
        y,
        conversion=exp_conversion,
        max_start_time_adjustment_factor=exp_max_start_time_adjustment_factor,
    )
    exp_fit_results[column] = fit

    output_path = output_dir / f"{column}_exp_fit_results.png"
    plot_exponential_fit(column, x, y, fit, output_path, prefix_label="Exponential")

    a_relative_error = relative_error(fit["A"], fit["A_err"])
    k_relative_error = relative_error(fit["k"], fit["k_err"])
    exp_reliable, exp_unreliable_reason = classify_exponential_fit(fit)

    exp_records.append(
        {
            "Entry": column,
            "exp_success": fit["success"],
            "exp_reliable": exp_reliable,
            "exp_failure_reason": fit["failure_reason"],
            "exp_unreliable_reason": exp_unreliable_reason,
            "A": fit["A"],
            "A_err": fit["A_err"],
            "A_relative_error": a_relative_error,
            "k_exp": fit["k"],
            "k_exp_err": fit["k_err"],
            "k_relative_error": k_relative_error,
            "exp_max_plausible_k": exp_max_plausible_k,
            "C": fit["C"],
            "C_err": fit["C_err"],
            "R2_exp": fit["R2"],
            "chosen_region": fit["chosen_region"],
            "chosen_fit_start_time": fit["chosen_fit_start_time"],
            "chosen_fit_end_time": fit["chosen_fit_end_time"],
            "precise_fit_start_time": fit["precise_fit_start_time"],
            "precise_fit_end_time": fit["precise_fit_end_time"],
            "t_conv": fit["t_conv"],
            "plot_file": str(output_path) if save_plots else "",
        }
    )

exp_results_df = pd.DataFrame(exp_records)
print(exp_results_df.head())


## Part VI: Exponential quality check

Calculate residual-based quality metrics for the exponential fits. These values are reported as inspection warnings but do not by themselves determine the final fitting method.

In [ ]:
quality_records = []

for column, fit in exp_fit_results.items():
    record = {
        "Entry": column,
        "absorbance_change": np.nan,
        "max_residual_deviation": np.nan,
        "percent_deviation": np.nan,
        "quality_flag": "fit_failed",
    }

    if fit["success"]:
        residuals = fit["y_precise"] - fit["y_precise_pred"]
        absorbance_change = fit["y_precise"][-1] - fit["y_precise"][0]
        max_residual_deviation = np.abs(residuals).max()
        percent_deviation = (
            (max_residual_deviation / absorbance_change) * 100
            if absorbance_change != 0
            else np.nan
        )

        if np.isfinite(percent_deviation) and abs(percent_deviation) > 5:
            quality_flag = "low_quality_check"
        elif np.isfinite(percent_deviation) and abs(percent_deviation) > 2:
            quality_flag = "slight_deviation_check"
        else:
            quality_flag = "ok"

        record.update(
            {
                "absorbance_change": absorbance_change,
                "max_residual_deviation": max_residual_deviation,
                "percent_deviation": percent_deviation,
                "quality_flag": quality_flag,
            }
        )

    quality_records.append(record)

exp_quality_df = pd.DataFrame(quality_records)
exp_quality_lookup = exp_quality_df.set_index("Entry")["quality_flag"].to_dict()
print(exp_quality_df["quality_flag"].value_counts(dropna=False))
print(exp_quality_df.head())


## Part VII: Plate-level A value for initial-rate conversion

Determine the plate-level amplitude from reliable exponential fits. The median `A` value is used to convert absorbance slopes into first-order rate constants in Part VIII.

In [ ]:
def bootstrap_median_ci(values, iterations=10000, random_seed=42, alpha=0.05):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan
    if len(values) == 1:
        return values[0], values[0]

    rng = np.random.default_rng(random_seed)
    boot = np.empty(iterations)
    for i in range(iterations):
        sample = rng.choice(values, size=len(values), replace=True)
        boot[i] = np.median(sample)

    lower = np.percentile(boot, 100 * alpha / 2)
    upper = np.percentile(boot, 100 * (1 - alpha / 2))
    return lower, upper


reliable_a_df = exp_results_df[
    (exp_results_df["exp_reliable"])
    & np.isfinite(exp_results_df["A"])
    & np.isfinite(exp_results_df["A_err"])
    & (exp_results_df["A"] != 0)
    & (exp_results_df["A_relative_error"] < a_relative_error_cutoff)
].copy()

a_values = reliable_a_df["A"].astype(float).values
a_plate_median = float(np.median(a_values)) if len(a_values) else np.nan
a_plate_ci_low, a_plate_ci_high = bootstrap_median_ci(
    a_values,
    iterations=bootstrap_iterations,
    random_seed=bootstrap_random_seed,
)

a_summary_df = pd.DataFrame(
    [
        {
            "n_reliable_A_values": len(a_values),
            "minimum_requested_reliable_A_values": minimum_reliable_a_values,
            "A_relative_error_cutoff": a_relative_error_cutoff,
            "A_plate_median": a_plate_median,
            "A_plate_median_CI95_low": a_plate_ci_low,
            "A_plate_median_CI95_high": a_plate_ci_high,
            "A_plate_mean": float(np.mean(a_values)) if len(a_values) else np.nan,
            "A_plate_sd": float(np.std(a_values, ddof=1)) if len(a_values) > 1 else np.nan,
            "enough_reliable_A_values": len(a_values) >= minimum_reliable_a_values,
        }
    ]
)

print(a_summary_df.to_string(index=False))
if len(a_values) < minimum_reliable_a_values:
    print(
        "Warning: fewer than the requested number of reliable A values were found. "
        "Inspect A_relative_error and the exponential fits before using initial-rate k_obs."
    )


## Part VIII: Initial-rate analysis

For each trace, the notebook first tries to calculate the initial absorbance slope from the analytical derivative of a reliable exponential fit at the actual start of the reaction:

`initial_slope = A * k`  (absorbance increase; the decrease notebook uses `initial_slope = -A * k`)

This slope is converted to a first-order rate constant using the plate-level median amplitude from step VII:

`k_obs_init = abs(initial_slope) / abs(A_plate_median)`

If the exponential parameters are not reliable, the notebook uses a linear fallback fit. The fallback may remove up to the first 5% of the trace to avoid mixing artifacts (see definitions in Part I), but it must still cover at least 50% of the data.

In [ ]:
def plot_initial_rate_fit(column, x, y, exp_fit, linear_fit, method, output_path):
    fig, (ax1, ax2, ax3) = plt.subplots(
        3, 1, figsize=(8, 15), gridspec_kw={"height_ratios": [3, 3, 1]}
    )
    fig.suptitle(f"Initial-rate analysis for {column}", y=0.925)

    ax1.plot(x, y, label="Original Data")
    ax2.plot(x, y, label="Original Data")

    if method == "exp_derivative":
        slope = exp_fit["A"] * exp_fit["k"]
        y0 = exp_fit["C"]
        tangent_y = y0 + slope * x

        ax1.plot(
            exp_fit["x_precise"],
            exp_fit["y_precise_pred"],
            label="Exponential fit used for derivative",
            color="red",
        )
        ax1.plot(x, tangent_y, label=f"Tangent at t=0 (slope={slope:.3e})", color="green", linestyle="--")
        ax1.fill_betweenx(
            ax1.get_ylim(),
            exp_fit["precise_fit_start_time"],
            exp_fit["precise_fit_end_time"],
            color="gray",
            alpha=0.2,
            label="Fit Region",
        )

        ax2.plot(
            exp_fit["x_precise"],
            exp_fit["y_precise_pred"],
            label=(
                f"Exp derivative (A={exp_fit['A']:.4f}, "
                f"k={exp_fit['k']:.7f}, R2={exp_fit['R2']:.4f})"
            ),
            color="red",
        )
        ax2.plot(x, tangent_y, label=f"Tangent at t=0", color="green", linestyle="--")
        ax2.fill_betweenx(
            ax2.get_ylim(),
            exp_fit["precise_fit_start_time"],
            exp_fit["precise_fit_end_time"],
            color="gray",
            alpha=0.2,
            label="Fit Region",
        )
        ax2.set_xlim(exp_fit["precise_fit_start_time"], exp_fit["precise_fit_end_time"])

        residual_x = exp_fit["x_precise"]
        residuals = exp_fit["y_precise"] - exp_fit["y_precise_pred"]
        ax3.plot(residual_x, residuals, label="Residuals", color="purple")
        ax3.set_xlim(exp_fit["precise_fit_start_time"], exp_fit["precise_fit_end_time"])

    else:
        ax1.plot(
            linear_fit["x_fit"],
            linear_fit["y_pred"],
            label=f"Linear fallback (slope={linear_fit['slope']:.3e}, R2={linear_fit['R2']:.4f})",
            color="red",
        )
        ax1.fill_betweenx(
            ax1.get_ylim(),
            linear_fit["fit_start_time"],
            linear_fit["fit_end_time"],
            color="gray",
            alpha=0.2,
            label="Linear Fit Region",
        )

        ax2.plot(
            linear_fit["x_fit"],
            linear_fit["y_pred"],
            label=f"Linear fallback (slope={linear_fit['slope']:.3e}, R2={linear_fit['R2']:.4f})",
            color="red",
        )
        ax2.fill_betweenx(
            ax2.get_ylim(),
            linear_fit["fit_start_time"],
            linear_fit["fit_end_time"],
            color="gray",
            alpha=0.2,
            label="Linear Fit Region",
        )
        ax2.set_xlim(linear_fit["fit_start_time"], linear_fit["fit_end_time"])

        residuals = linear_fit["y_fit"] - linear_fit["y_pred"]
        ax3.plot(linear_fit["x_fit"], residuals, label="Residuals", color="purple")
        ax3.set_xlim(linear_fit["fit_start_time"], linear_fit["fit_end_time"])

    for ax in (ax1, ax2):
        ax.set_xlabel("Time [s]")
        ax.set_ylabel("Value")
        ax.legend()

    ax3.axhline(y=0, color="black", linestyle="--")
    ax3.set_xlabel("Time [s]")
    ax3.set_ylabel("Residual")
    ax3.legend()

    if save_plots:
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
    if show_plots:
        plt.show()
    else:
        plt.close(fig)


init_records = []
exp_result_lookup = exp_results_df.set_index("Entry").to_dict("index")

plate_a_available = np.isfinite(a_plate_median) and a_plate_median != 0
if not plate_a_available:
    print(
        "Warning: A_plate_median is invalid. Initial slopes will be reported, "
        "but k_init cannot be calculated for this dataset."
    )

for column in analysis_columns:
    x = data_frame["Time"].values
    y = data_frame[column].values
    exp_fit = exp_fit_results[column]
    exp_row = exp_result_lookup.get(column, {})
    exp_quality_flag = exp_quality_lookup.get(column, "fit_failed")
    linear_fit = fit_linear_roi(
        x,
        y,
        min_region_fraction=minimum_linear_region_fraction,
        start_fraction_candidates=linear_start_fraction_candidates,
        end_fraction_candidates=linear_end_fraction_candidates,
    )

    exp_slope = (
        exp_fit["A"] * exp_fit["k"]
        if exp_fit["success"] and np.isfinite(exp_fit["A"]) and np.isfinite(exp_fit["k"]) and exp_fit["k"] > 0
        else np.nan
    )
    exp_slope_err = (
        math.sqrt((exp_fit["k"] * exp_fit["A_err"]) ** 2 + (exp_fit["A"] * exp_fit["k_err"]) ** 2)
        if np.isfinite(exp_slope)
        and np.isfinite(exp_fit["A_err"])
        and np.isfinite(exp_fit["k_err"])
        else np.nan
    )

    linear_slope = linear_fit["slope"] if linear_fit["success"] else np.nan
    linear_slope_err = linear_fit["slope_err"] if linear_fit["success"] else np.nan

    exp_a_relative_error = relative_error(exp_fit["A"], exp_fit["A_err"])
    exp_derivative_allowed = (
        bool(exp_row.get("exp_reliable", False))
        and np.isfinite(exp_slope)
        and np.isfinite(exp_a_relative_error)
        and exp_a_relative_error < initial_rate_exp_a_relative_error_cutoff
    )

    if exp_derivative_allowed:
        method = "exp_derivative"
        selected_slope = exp_slope
        selected_slope_err = exp_slope_err
    elif np.isfinite(linear_slope):
        method = "linear_fallback"
        selected_slope = linear_slope
        selected_slope_err = linear_slope_err
    else:
        method = "failed"
        selected_slope = np.nan
        selected_slope_err = np.nan

    k_init = (
        abs(selected_slope) / abs(a_plate_median)
        if plate_a_available and np.isfinite(selected_slope)
        else np.nan
    )
    k_init_err = (
        abs(selected_slope_err) / abs(a_plate_median)
        if plate_a_available and np.isfinite(selected_slope_err)
        else np.nan
    )

    output_path = output_dir / f"{column}_init_fit_results.png"
    if method != "failed":
        plot_initial_rate_fit(column, x, y, exp_fit, linear_fit, method, output_path)

    init_records.append(
        {
            "Entry": column,
            "init_method": method,
            "plate_A_available": plate_a_available,
            "A_plate_median": a_plate_median,
            "initial_slope": selected_slope,
            "initial_slope_err": selected_slope_err,
            "k_init": k_init,
            "k_init_err": k_init_err,
            "exp_reliable_for_derivative": bool(exp_row.get("exp_reliable", False)),
            "exp_quality_flag": exp_quality_flag,
            "exp_derivative_allowed": exp_derivative_allowed,
            "exp_A_relative_error": exp_a_relative_error,
            "initial_rate_exp_A_relative_error_cutoff": initial_rate_exp_a_relative_error_cutoff,
            "exp_derivative_slope": exp_slope,
            "exp_derivative_slope_err": exp_slope_err,
            "linear_slope": linear_slope,
            "linear_slope_err": linear_slope_err,
            "linear_R2": linear_fit["R2"],
            "linear_chosen_region": linear_fit["chosen_region"],
            "linear_start_fraction": linear_fit["chosen_start_fraction"],
            "linear_end_fraction": linear_fit["chosen_end_fraction"],
            "linear_fit_start_time": linear_fit["fit_start_time"],
            "linear_fit_end_time": linear_fit["fit_end_time"],
            "plot_file": str(output_path) if method != "failed" and save_plots else "",
        }
    )

init_results_df = pd.DataFrame(init_records)
print(init_results_df.head())
print(init_results_df["init_method"].value_counts(dropna=False))


## Part IX: Compare exponential and initial-rate results

Compare exponential and initial-rate-derived rate constants using the `k_max` threshold defined in part I. The final `k_obs` value and the method used to obtain it are selected for each well.

In [ ]:
x = data_frame["Time"].values.astype(float)
t_total = float(x[-1])
k_max = float(np.log(2) / (2 * t_total))

comparison_df = exp_results_df[
    [
        "Entry",
        "k_exp",
        "k_exp_err",
        "exp_success",
        "exp_reliable",
        "exp_unreliable_reason",
        "R2_exp",
        "A_relative_error",
        "k_relative_error",
        "exp_max_plausible_k",
    ]
].merge(
    exp_quality_df[["Entry", "quality_flag"]],
    on="Entry",
    how="left",
).merge(
    init_results_df[
        ["Entry", "k_init", "k_init_err", "init_method"]
    ],
    on="Entry",
    how="left",
)

comparison_df["t_total"] = t_total
comparison_df["k_max"] = k_max
comparison_df["exp_faster_than_k_max"] = np.where(
    comparison_df["k_exp"] > comparison_df["k_max"],
    "YES",
    "NO",
)
comparison_df["exp_faster_flag"] = np.where(
    comparison_df["k_exp"] > comparison_df["k_max"],
    1,
    0,
)

comparison_df["exp_usable_for_final_kobs"] = (
    comparison_df["exp_reliable"].astype(bool)
    & (comparison_df["exp_faster_flag"] == 1)
)
comparison_df["init_usable_for_final_kobs"] = (
    np.isfinite(comparison_df["k_init"])
    & comparison_df["init_method"].notna()
    & (comparison_df["init_method"] != "failed")
)

comparison_df["k_obs"] = np.select(
    [
        comparison_df["exp_usable_for_final_kobs"],
        comparison_df["init_usable_for_final_kobs"],
    ],
    [
        comparison_df["k_exp"],
        comparison_df["k_init"],
    ],
    default=np.nan,
)
comparison_df["k_obs_method"] = np.select(
    [
        comparison_df["exp_usable_for_final_kobs"],
        comparison_df["init_usable_for_final_kobs"],
    ],
    [
        "exponential",
        "initial_rate_" + comparison_df["init_method"].astype(str),
    ],
    default="discarded_outlier",
)
comparison_df["k_init_over_k_exp"] = comparison_df["k_init"] / comparison_df["k_exp"]

# Fitting statistics summary.
k_obs_method_counts = comparison_df["k_obs_method"].value_counts(dropna=False)

fitting_statistics_txt = output_dir / f"{PROJECT_FOLDER_NAME}_fitting_statistics.txt"

with open(fitting_statistics_txt, "w") as f:
    f.write("Fitting statistics\n")
    f.write("==================\n\n")
    f.write(f"Dataset: {PROJECT_FOLDER_NAME}\n")
    f.write(f"Input file: {input_excel.name}\n")
    f.write(f"t_total = {t_total:.1f} s\n")
    f.write(f"k_max = {k_max:.6e} s^-1\n")
    f.write(f"exp_max_plausible_k = {exp_max_plausible_k}\n\n")

    f.write("Final k_obs method counts:\n")
    f.write(k_obs_method_counts.to_string())
    f.write("\n\n")

    f.write("Exponential reliability counts:\n")
    f.write(comparison_df["exp_reliable"].value_counts(dropna=False).to_string())
    f.write("\n\n")

    f.write("Exponential unreliable reasons:\n")
    f.write(comparison_df["exp_unreliable_reason"].fillna("reliable").value_counts(dropna=False).to_string())
    f.write("\n")

print(f"t_total = {t_total:.1f} s")
print(f"k_max = {k_max:.6e} s^-1")
print(comparison_df.head())
print(k_obs_method_counts)
print(f"Saved fitting statistics to {fitting_statistics_txt}")


## Part X: Save all fitted parameters

Export all fit parameters, quality metrics, plate-level `A` values, initial-rate results, and final selected `k_obs` values to an Excel workbook.

In [ ]:
with pd.ExcelWriter(fit_results_excel, engine="openpyxl") as writer:
    exp_results_df.to_excel(writer, sheet_name="Exponential_Fits", index=False)
    exp_quality_df.to_excel(writer, sheet_name="Exp_Quality_Check", index=False)
    reliable_a_df.to_excel(writer, sheet_name="Reliable_A_Values", index=False)
    a_summary_df.to_excel(writer, sheet_name="Plate_A_Summary", index=False)
    init_results_df.to_excel(writer, sheet_name="Initial_Rates", index=False)
    comparison_df.to_excel(writer, sheet_name="Comparison_Final_kobs", index=False)

print(f"Saved fitted parameters and comparison tables to {fit_results_excel}")
